## Section 0 — Setup

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv
/kaggle/input/pm-120630246-at-07-24-2026-15-04-07/__script__.py
/kaggle/input/pm-120630246-at-07-24-2026-15-04-07/annotated_types-0.8.0-py3-none-any.whl
/kaggle/input/pm-120630246-at-07-24-2026-15-04-07/requests-2.34.2-py3-none-any.whl
/kaggle/input/pm-120630246-at-07-24-2026-15-04-07/platformdirs-4.11.0-py3-none-any.whl
/kaggle/input/pm-120630246-at-07-24-2026-15-04-07/pydantic_core-2.46.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl
/kaggle/input/pm-120630246-at-07-24-2026-15-04-07/idna-3.18-py3-none-any.whl
/kaggle/input/pm-120630246-at-07-24-2026-15-04-07/protobuf-7.35.1-cp310-abi3-manylinux2014_x86_64.whl
/kaggle/input/pm-120630246-at-07-24-2026-15-04-07/click-8.4.2-py3-none-any.whl
/kaggle/input/pm-120630246-at-07-24-2026-15-04-07/__results__.html
/kaggle/input/pm-12

In [2]:
!pip install -q -U transformers datasets peft sentence-transformers faiss-cpu wandb scikit-learn lightgbm torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 95.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 35.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 38.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 596.7/596.7 kB 39.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 88.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.5/26.5 MB 72.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 106.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 105.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 79.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 771.9/771.9 kB 43.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.2/119.2 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 29.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4

In [3]:
import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, accuracy_score

# ---- Project config -------------------------------------------------------
ROLL_NO = "24f2003563"
WANDB_PROJECT = f"{ROLL_NO}-t22026"          # must match the W&B project you registered
DATA_DIR = "/kaggle/input/competitions/smart-mcq-solver-challenge"
OUT_DIR = "/kaggle/working"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


Device: cuda


In [4]:
train = pd.read_csv(f"{DATA_DIR}/train.csv")
test  = pd.read_csv(f"{DATA_DIR}/test.csv")

print("Train:", train.shape, "| Test:", test.shape)
train.head()


Train: (2000, 8) | Test: (500, 7)


,id,prompt,A,B,C,D,E,answer
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C
3,4,Select the most accurate option: What is Marti...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B
4,5,Identify the correct statement: What is the co...,"Simultaneity is relative, meaning that two eve...","Simultaneity is relative, meaning that two eve...","Simultaneity is absolute, meaning that two eve...",Simultaneity is a concept that applies only to...,Simultaneity is a concept that applies only to...,A


In [5]:
# Quick sanity checks worth putting in the report's "Dataset" section.
print("Missing values in train:\n", train.isnull().sum().sum())
print("Answer options seen:", sorted(train['answer'].unique()))
print("Every question has exactly 5 options (A-E), 1 of which is correct ->",
      "expected positive rate per row ≈ 0.20")

Missing values in train:
 0
Answer options seen: ['A', 'B', 'C', 'D', 'E']
Every question has exactly 5 options (A-E), 1 of which is correct -> expected positive rate per row ≈ 0.20


In [6]:
def build_dataset(df, has_answer = True):
    """Explode each question into 5 rows (one per option A-E)."""
    rows = []
    for _, row in df.iterrows():
        for option in ['A', 'B', 'C', 'D', 'E']:
            item = {
                "id": row['id'],
                "prompt": row['prompt'],
                "option_text": row[option],
                "text": f"Question: {row['prompt']} Candidate Answer: {row[option]}",
                "option": option,
            }
            if has_answer:
                item["label"] = 1 if option == row['answer'] else 0
            rows.append(item)
    return pd.DataFrame(rows)

train_processed = build_dataset(train, has_answer=True)
test_processed  = build_dataset(test,  has_answer=False)
print(train_processed.shape, test_processed.shape)

(10000, 6) (2500, 5)


In [7]:
# Split by QUESTION id (not row!) so all 5 options of a question stay together.
train_ids, val_ids = train_test_split(train['id'].unique(), test_size=0.15, random_state=42)

train_df = train_processed[train_processed['id'].isin(train_ids)].reset_index(drop=True)
val_df   = train_processed[train_processed['id'].isin(val_ids)].reset_index(drop=True)
val_answers = train[train['id'].isin(val_ids)].set_index("id")["answer"]

print(f"Train questions: {len(train_ids)} | Val questions: {len(val_ids)}")


Train questions: 1700 | Val questions: 300


In [8]:
def map_at_3(true_answers_by_id, submission_df):
    """Standard MAP@3, the official Kaggle leaderboard metric for this competition."""
    scores = []
    for _, row in submission_df.iterrows():
        true = true_answers_by_id.loc[row["ID"]]
        preds = row["Prediction"].split()[:3]
        score = 0.0
        for i, p in enumerate(preds):
            if p == true:
                score = 1.0 / (i + 1)
                break
        scores.append(score)
    return float(np.mean(scores))


def scores_to_submission(id_list, option_list, score_list, out_path = None):
    """Turn per-option scores into a ranked top-3 submission (ID, Prediction)."""
    df = pd.DataFrame({"id": id_list, "option": option_list, "score": score_list})
    rows = [
        {"ID": qid, "Prediction": " ".join(g.sort_values("score", ascending=False)["option"].values[:3])}
        for qid, g in df.groupby("id")
    ]
    sub = pd.DataFrame(rows)
    if out_path:
        sub.to_csv(out_path, index=False)
    return sub


def normalize(scores):
    """Min-max scale so scores from very different models can be averaged fairly."""
    scores = np.array(scores, dtype=float)
    return (scores - scores.min()) / (scores.max() - scores.min() + 1e-8)


def evaluate_model(name, df, scores, true_answers_by_id):
    """
    One evaluation function for every model in this notebook.

    - MAP@3       : official competition metric (top-3 ranking)
    - Accuracy/F1 : treat the highest-scored option per question as the model's
                    single prediction, and compare it to the true label column.
    """
    work = df.reset_index(drop=True).copy()
    work["score"] = np.asarray(scores)

    sub = scores_to_submission(work["id"].values, work["option"].values, work["score"].values)
    map3 = map_at_3(true_answers_by_id, sub)

    work["pred_label"] = 0
    top_idx = work.groupby("id")["score"].idxmax()
    work.loc[top_idx, "pred_label"] = 1

    acc = accuracy_score(work["label"], work["pred_label"])
    f1 = f1_score(work["label"], work["pred_label"], zero_division=0)

    metrics = {"val_map3": map3, "val_accuracy": acc, "val_f1": f1}
    print(f"{name:32s} | MAP@3={map3:.4f} | Accuracy={acc:.4f} | F1={f1:.4f}")
    return metrics


## W&B Login

In [9]:
import wandb
from kaggle_secrets import UserSecretsClient

wandb.login(key=UserSecretsClient().get_secret("WANDB_API_KEY"))

def log_run(run_name, metrics, config = None):
    """Log one finished model's metrics as its own W&B run (keeps runs comparable)."""
    run = wandb.init(project=WANDB_PROJECT, name=run_name, config=config or {})
    wandb.log(metrics)
    run.finish()


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: 24f2003563 (24f2003563-iit-madras) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


## TF-IDF / Word2Vec Baseline + Cosine Similarity


In [10]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

tfidf = TfidfVectorizer(max_features=20000, ngram_range=(1, 2), sublinear_tf=True)
X_train = tfidf.fit_transform(train_df["text"])
X_val   = tfidf.transform(val_df["text"])
X_test  = tfidf.transform(test_processed["text"])

logreg = LogisticRegression(max_iter=1000, class_weight="balanced")
logreg.fit(X_train, train_df["label"])

val_probs_tfidf  = logreg.predict_proba(X_val)[:, 1]
test_probs_tfidf = logreg.predict_proba(X_test)[:, 1]

metrics_tfidf = evaluate_model("Model 1 - TFIDF+LogReg", val_df, val_probs_tfidf, val_answers)
log_run("model1-tfidf-logreg", metrics_tfidf, config={"max_features": 20000, "ngram_range": "(1,2)"})


Model 1 - TFIDF+LogReg           | MAP@3=0.9917 | Accuracy=0.9947 | F1=0.9867


wandb: setting up run 5ebj8318
wandb: Tracking run with wandb version 0.28.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260724_150548-5ebj8318
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run model1-tfidf-logreg
wandb: ⭐️ View project at https://wandb.ai/24f2003563-iit-madras/24f2003563-t22026
wandb: 🚀 View run at https://wandb.ai/24f2003563-iit-madras/24f2003563-t22026/runs/5ebj8318
wandb: updating run metadata; uploading summary
wandb: uploading summary
wandb: uploading history steps 0-0, summary
wandb: 
wandb: Run history:
wandb: val_accuracy ▁
wandb:       val_f1 ▁
wandb:     val_map3 ▁
wandb: 
wandb: Run summary:
wandb: val_accuracy 0.99467
wandb:       val_f1 0.98667
wandb:     val_map3 0.99167
wandb: 
wandb: 🚀 View run model1-tfidf-logreg at: https://wandb.ai/24f2003563-iit-madras/24f2003563-t22026/runs/5ebj8318
wandb: ⭐️ View project at: https://wandb.ai/24f2003563-iit-madras/24f2003563-t22026
wandb: Synced 4 W&B file(s), 0 media file(s), 

## Transformer Embeddings + Zero-Shot Classification

In [11]:
from sentence_transformers import SentenceTransformer

embed_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=str(device))

def zero_shot_score(df):
    p_emb = embed_model.encode(df["prompt"].tolist(), batch_size=64, convert_to_numpy=True)
    o_emb = embed_model.encode(df["option_text"].tolist(), batch_size=64, convert_to_numpy=True)
    return np.sum(p_emb * o_emb, axis=1) / (
        np.linalg.norm(p_emb, axis=1) * np.linalg.norm(o_emb, axis=1) + 1e-8
    )

val_scores_zeroshot  = zero_shot_score(val_df)
test_scores_zeroshot = zero_shot_score(test_processed)

metrics_zeroshot = evaluate_model("Model 2 - Zero-shot MiniLM", val_df, val_scores_zeroshot, val_answers)
log_run("model2-zeroshot-minilm", metrics_zeroshot, config={"model": "all-MiniLM-L6-v2"})


Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model 2 - Zero-shot MiniLM       | MAP@3=0.4050 | Accuracy=0.7040 | F1=0.2600


wandb: setting up run c63me28x
wandb: Tracking run with wandb version 0.28.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260724_150635-c63me28x
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run model2-zeroshot-minilm
wandb: ⭐️ View project at https://wandb.ai/24f2003563-iit-madras/24f2003563-t22026
wandb: 🚀 View run at https://wandb.ai/24f2003563-iit-madras/24f2003563-t22026/runs/c63me28x
wandb: updating run metadata; uploading summary
wandb: uploading summary
wandb: uploading history steps 0-0, summary
wandb: 
wandb: Run history:
wandb: val_accuracy ▁
wandb:       val_f1 ▁
wandb:     val_map3 ▁
wandb: 
wandb: Run summary:
wandb: val_accuracy 0.704
wandb:       val_f1 0.26
wandb:     val_map3 0.405
wandb: 
wandb: 🚀 View run model2-zeroshot-minilm at: https://wandb.ai/24f2003563-iit-madras/24f2003563-t22026/runs/c63me28x
wandb: ⭐️ View project at: https://wandb.ai/24f2003563-iit-madras/24f2003563-t22026
wandb: Synced 4 W&B file(s), 0 media file(s), 0

# RAG 

In [12]:
import faiss

corpus_texts = train["prompt"].tolist()
corpus_emb = embed_model.encode(corpus_texts, batch_size=64, convert_to_numpy=True)
corpus_emb = corpus_emb / (np.linalg.norm(corpus_emb, axis=1, keepdims=True) + 1e-8)

index = faiss.IndexFlatIP(corpus_emb.shape[1])
index.add(corpus_emb.astype("float32"))

def retrieve_context(query, k = 2):
    q_emb = embed_model.encode([query], convert_to_numpy=True)
    q_emb = q_emb / (np.linalg.norm(q_emb, axis=1, keepdims=True) + 1e-8)
    _, idxs = index.search(q_emb.astype("float32"), k)
    return " | ".join(corpus_texts[i] for i in idxs[0])

def rag_score(df, k = 2):
    rag_text = [f"Context: {retrieve_context(p, k)} Question: {p}" for p in df["prompt"]]
    ctx_emb = embed_model.encode(rag_text, batch_size=32, convert_to_numpy=True)
    opt_emb = embed_model.encode(df["option_text"].tolist(), batch_size=32, convert_to_numpy=True)
    return np.sum(ctx_emb * opt_emb, axis=1) / (
        np.linalg.norm(ctx_emb, axis=1) * np.linalg.norm(opt_emb, axis=1) + 1e-8
    )

val_rag_scores  = rag_score(val_df)
test_rag_scores = rag_score(test_processed)

metrics_rag = evaluate_model("Bonus - RAG-augmented", val_df, val_rag_scores, val_answers)
log_run("bonus-rag-signal", metrics_rag, config={"k_retrieved": 2})


Bonus - RAG-augmented            | MAP@3=0.4061 | Accuracy=0.7067 | F1=0.2667


wandb: Tracking run with wandb version 0.28.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260724_150742-6mn0j1q4
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run bonus-rag-signal
wandb: ⭐️ View project at https://wandb.ai/24f2003563-iit-madras/24f2003563-t22026
wandb: 🚀 View run at https://wandb.ai/24f2003563-iit-madras/24f2003563-t22026/runs/6mn0j1q4
wandb: updating run metadata; uploading summary
wandb: uploading summary; uploading wandb-metadata.json; uploading requirements.txt; uploading wandb-summary.json; uploading config.yaml
wandb: uploading summary
wandb: uploading history steps 0-0, summary
wandb: 
wandb: Run history:
wandb: val_accuracy ▁
wandb:       val_f1 ▁
wandb:     val_map3 ▁
wandb: 
wandb: Run summary:
wandb: val_accuracy 0.70667
wandb:       val_f1 0.26667
wandb:     val_map3 0.40611
wandb: 
wandb: 🚀 View run bonus-rag-signal at: https://wandb.ai/24f2003563-iit-madras/24f2003563-t22026/runs/6mn0j1q4
wandb: ⭐️ View project at: htt

## LoRA Fine-Tuning (DeBERTa-v3)

In [13]:
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
TrainingArguments, Trainer, EarlyStoppingCallback
)
from peft import LoraConfig, get_peft_model, TaskType
from datasets import Dataset
from torch.utils.data import Dataset as TorchDataset

MODEL_NAME = "microsoft/deberta-v3-large"
MAX_LEN = 256

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=MAX_LEN)

class MCQDataset(TorchDataset):
    """Plain PyTorch dataset (avoids a known HF/torchvision bug with .set_format)."""
    def __init__(self, hf_dataset):
        d = hf_dataset.to_dict()
        self.input_ids = torch.tensor(d["input_ids"])
        self.attention_mask = torch.tensor(d["attention_mask"])
        self.token_type_ids = torch.tensor(d["token_type_ids"]) if "token_type_ids" in d else None
        self.labels = torch.tensor(d["labels"]) if "labels" in d else None

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, i):
        item = {"input_ids": self.input_ids[i], "attention_mask": self.attention_mask[i]}
        if self.token_type_ids is not None:
            item["token_type_ids"] = self.token_type_ids[i]
        if self.labels is not None:
            item["labels"] = self.labels[i]
        return item

hf_train = Dataset.from_pandas(train_df[["text", "label"]]).map(tokenize, batched=True).rename_column("label", "labels")
hf_val   = Dataset.from_pandas(val_df[["text", "label"]]).map(tokenize, batched=True).rename_column("label", "labels")

torch_train = MCQDataset(hf_train)
torch_val   = MCQDataset(hf_val)


config.json:   0%|          | 0.00/580 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

spm.model: reconstructing file:   0%|          |  0.00B / 2.46MB            

spm.model: downloading bytes:           |  0.00B            

Map:   0%|          | 0/8500 [00:00<?, ? examples/s]

Map:   0%|          | 0/1500 [00:00<?, ? examples/s]

In [14]:
class WeightedTrainer(Trainer):
    """Weights the rare 'correct answer' class higher (1 correct : 4 wrong per question)."""
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits.float()
        weight = torch.tensor([1.0, 4.0]).to(logits.device)
        loss = nn.CrossEntropyLoss(weight=weight)(logits, labels)
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds, zero_division=0),
        "pred_positive_rate": preds.mean(),   # sanity check: should land near 0.2
    }


In [15]:
base_model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(device)

lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=32,
    lora_alpha=64,
    lora_dropout=0.1,
    target_modules=["query_proj", "key_proj", "value_proj", "dense"],
)
lora_model = get_peft_model(base_model, lora_config)
lora_model.print_trainable_parameters()

EPOCHS = 8
steps_per_epoch = max(1, len(torch_train) // (8 * 4))
warmup_steps = int(0.1 * steps_per_epoch * EPOCHS)

training_args = TrainingArguments(
    output_dir=f"{OUT_DIR}/lora_outputs",
    learning_rate=2e-4,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=4,
    num_train_epochs=EPOCHS,
    weight_decay=0.01,
    max_grad_norm=1.0,
    warmup_steps=warmup_steps,
    fp16=False, bf16=False,
    dataloader_pin_memory=False,
    dataloader_num_workers=0,
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    report_to="wandb",
    run_name="model3-deberta-lora",
)

trainer = WeightedTrainer(
    model=lora_model,
    args=training_args,
    train_dataset=torch_train,
    eval_dataset=torch_val,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

trainer.train()


pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  874MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/390 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  874MB            

model.safetensors: downloading bytes:           |  0.00B            

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-large
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
classifier.weight                       | MISSING    | 
pooler.dense.bias                       | MISSING    | 
pooler.dense.weight                     | MISSING    | 
classifie

trainable params: 14,223,362 || all params: 449,287,172 || trainable%: 3.1658


wandb: Tracking run with wandb version 0.28.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260724_150803-npcj0qe9
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run model3-deberta-lora
wandb: ⭐️ View project at https://wandb.ai/24f2003563-iit-madras/huggingface
wandb: 🚀 View run at https://wandb.ai/24f2003563-iit-madras/huggingface/runs/npcj0qe9
[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!
[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


Epoch,Training Loss,Validation Loss,Accuracy,F1,Pred Positive Rate
1,2.778165,0.683919,0.753333,0.222689,0.117333
2,1.770069,0.227811,0.898000,0.788966,0.283333
3,0.611790,0.134248,0.986667,0.965986,0.192000
4,0.263938,0.064738,0.986667,0.967320,0.208000
5,0.193775,0.012427,0.996000,0.989967,0.198667
6,0.111812,0.042883,0.995333,0.988275,0.198000
7,0.031077,0.003501,0.997333,0.993377,0.202667
8,0.005404,0.001165,1.000000,1.000000,0.200000


TrainOutput(global_step=1064, training_loss=0.7436482850508151, metrics={'train_runtime': 3851.9646, 'train_samples_per_second': 17.653, 'train_steps_per_second': 0.276, 'total_flos': 3.3171481976832e+16, 'train_loss': 0.7436482850508151, 'epoch': 8.0})

In [16]:
def lora_score(dataset):
    """Run the fine-tuned model over a dataset and return P(option is correct)."""
    lora_model.eval()
    loader = torch.utils.data.DataLoader(dataset, batch_size=32)
    probs = []
    with torch.no_grad():
        for batch in loader:
            batch = {k: v.to(device) for k, v in batch.items() if k != "labels"}
            out = lora_model(**batch)
            probs.extend(F.softmax(out.logits.float(), dim=-1)[:, 1].cpu().numpy())
    return np.array(probs)

val_probs_lora = lora_score(torch_val)

metrics_lora = evaluate_model("Model 3 - DeBERTa+LoRA", val_df, val_probs_lora, val_answers)
log_run("model3-deberta-lora", metrics_lora, config={"model": MODEL_NAME, "lora_r": 32, "epochs": EPOCHS})


wandb: Finishing previous runs because reinit is set to 'default'.
wandb: uploading data; updating run metadata


Model 3 - DeBERTa+LoRA           | MAP@3=1.0000 | Accuracy=1.0000 | F1=1.0000


wandb: uploading data
wandb: 
wandb: Run history:
wandb:           eval/accuracy ▁▅██████
wandb:                 eval/f1 ▁▆██████
wandb:               eval/loss █▃▂▂▁▁▁▁
wandb: eval/pred_positive_rate ▁█▄▅▄▄▅▄
wandb:            eval/runtime ▇▃▇▁█▆██
wandb: eval/samples_per_second ▂▆▂█▁▃▁▁
wandb:   eval/steps_per_second ▂▆▂█▂▃▁▁
wandb:             train/epoch ▁▁▂▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▇▇▇▇████
wandb:       train/global_step ▁▁▂▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▇▇▇▇████
wandb:         train/grad_norm ▁▃▂▂█▃▂▁▃▅▂▁▁▁▁▁▁▁▁▁▁
wandb:                      +2 ...
wandb: 
wandb: Run summary:
wandb:           eval/accuracy 1
wandb:                 eval/f1 1
wandb:               eval/loss 0.00117
wandb: eval/pred_positive_rate 0.2
wandb:            eval/runtime 28.5813
wandb: eval/samples_per_second 52.482
wandb:   eval/steps_per_second 1.644
wandb:              total_flos 3.3171481976832e+16
wandb:             train/epoch 8
wandb:       train/global_step 1064
wandb:                      +7 ...
wandb: 
wandb: 🚀 View

## Milestone 5: Ensembling


In [17]:
summary = pd.DataFrame([
    {"model": "Model 1 - TF-IDF + LogisticRegression (from scratch)", **metrics_tfidf},
    {"model": "Model 2 - Zero-shot MiniLM embeddings (pretrained)",    **metrics_zeroshot},
    {"model": "Model 3 - DeBERTa-v3 + LoRA (fine-tuned)",              **metrics_lora},
]).sort_values("val_map3", ascending=False).reset_index(drop=True)

summary.to_csv(f"{OUT_DIR}/model_comparison.csv", index=False)

run = wandb.init(project=WANDB_PROJECT, name="model-comparison")
wandb.log({"comparison_table": wandb.Table(dataframe=summary)})
run.finish()

summary

wandb: Tracking run with wandb version 0.28.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260724_161305-krb149o2
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run model-comparison
wandb: ⭐️ View project at https://wandb.ai/24f2003563-iit-madras/24f2003563-t22026
wandb: 🚀 View run at https://wandb.ai/24f2003563-iit-madras/24f2003563-t22026/runs/krb149o2
wandb: updating run metadata; uploading artifact run-krb149o2-comparison_table; uploading summary
wandb: uploading artifact run-krb149o2-comparison_table
wandb: uploading history steps 0-0, summary
wandb: 🚀 View run model-comparison at: https://wandb.ai/24f2003563-iit-madras/24f2003563-t22026/runs/krb149o2
wandb: ⭐️ View project at: https://wandb.ai/24f2003563-iit-madras/24f2003563-t22026
wandb: Synced 4 W&B file(s), 1 media file(s), 2 artifact file(s) and 0 other file(s)
wandb: Find logs at: ./wandb/run-20260724_161305-krb149o2/logs


,model,val_map3,val_accuracy,val_f1
0,Model 3 - DeBERTa-v3 + LoRA (fine-tuned),1.000000,1.000000,1.000000
1,Model 1 - TF-IDF + LogisticRegression (from sc...,0.991667,0.994667,0.986667
2,Model 2 - Zero-shot MiniLM embeddings (pretrai...,0.405000,0.704000,0.260000


## Final Submission


In [18]:
best_model_scores = {
    "Model 1 - TF-IDF + LogisticRegression (from scratch)": val_probs_tfidf,
    "Model 2 - Zero-shot MiniLM embeddings (pretrained)": val_scores_zeroshot,
    "Model 3 - DeBERTa-v3 + LoRA (fine-tuned)": val_probs_lora,
}[summary.iloc[0]["model"]]

work = val_df.reset_index(drop=True).copy()
work["score"] = best_model_scores
top_idx = work.groupby("id")["score"].idxmax()
work["is_top_pick"] = False
work.loc[top_idx, "is_top_pick"] = True

wrong = work[(work["is_top_pick"]) & (work["label"] == 0)].head(5)
for _, row in wrong.iterrows():
    correct_option = val_answers.loc[row["id"]]
    print(f"Q: {row['prompt'][:120]}")
    print(f"   Model picked: {row['option']} ({row['option_text'][:60]})")
    print(f"   Correct was : {correct_option}")
    print()


In [19]:
hf_test = Dataset.from_pandas(test_processed[["text"]]).map(tokenize, batched=True)
torch_test = MCQDataset(hf_test)
test_probs_lora = lora_score(torch_test)

WEIGHTS = {"tfidf": 0.10, "zeroshot": 0.05, "rag": 0.20, "lora": 0.65}

val_ensemble = (
    WEIGHTS["tfidf"]    * normalize(val_probs_tfidf) +
    WEIGHTS["zeroshot"] * normalize(val_scores_zeroshot) +
    WEIGHTS["rag"]      * normalize(val_rag_scores) +
    WEIGHTS["lora"]     * normalize(val_probs_lora)
)
metrics_ensemble = evaluate_model("Ensemble (4 signals)", val_df, val_ensemble, val_answers)
log_run("final-ensemble", metrics_ensemble, config=WEIGHTS)

test_ensemble = (
    WEIGHTS["tfidf"]    * normalize(test_probs_tfidf) +
    WEIGHTS["zeroshot"] * normalize(test_scores_zeroshot) +
    WEIGHTS["rag"]      * normalize(test_rag_scores) +
    WEIGHTS["lora"]     * normalize(test_probs_lora)
)

final_submission = scores_to_submission(
    test_processed["id"].values,
    test_processed["option"].values,
    test_ensemble,
    out_path=f"{OUT_DIR}/submission.csv",
)
print("Saved submission.csv")
final_submission.head(10)


Map:   0%|          | 0/2500 [00:00<?, ? examples/s]

Ensemble (4 signals)             | MAP@3=1.0000 | Accuracy=1.0000 | F1=1.0000


wandb: Tracking run with wandb version 0.28.1
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260724_161422-czjojalr
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run final-ensemble
wandb: ⭐️ View project at https://wandb.ai/24f2003563-iit-madras/24f2003563-t22026
wandb: 🚀 View run at https://wandb.ai/24f2003563-iit-madras/24f2003563-t22026/runs/czjojalr
wandb: updating run metadata; uploading summary
wandb: uploading history steps 0-0, summary
wandb: 
wandb: Run history:
wandb: val_accuracy ▁
wandb:       val_f1 ▁
wandb:     val_map3 ▁
wandb: 
wandb: Run summary:
wandb: val_accuracy 1
wandb:       val_f1 1
wandb:     val_map3 1
wandb: 
wandb: 🚀 View run final-ensemble at: https://wandb.ai/24f2003563-iit-madras/24f2003563-t22026/runs/czjojalr
wandb: ⭐️ View project at: https://wandb.ai/24f2003563-iit-madras/24f2003563-t22026
wandb: Synced 4 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)
wandb: Find logs at: ./wandb/run-20260724_161

Saved submission.csv


,ID,Prediction
0,1,A B E
1,2,B C E
2,3,B A E
3,4,E C A
4,5,C A B
5,6,D A E
6,7,E D C
7,8,B E A
8,9,C D E
9,10,B C E
